# Sandbox - Hyper-Parameters optimisation

This notebook is used to prototype the Hyper-Parameters optimisation process, in all scenarios of use of the algorithm.

---

## Imports & Config

In [1]:
from pygments.formatters import load_formatter_from_file
! pwd

/Users/simonlejoly/Documents/Work/mimosa/tests


In [2]:
! export XLA_PYTHON_CLIENT_MEM_FRACTION=.25

In [3]:
# Jax configuration
USE_JIT = True
USE_X64 = True
DEBUG_NANS = False
VERBOSE = False

In [4]:
# Standard library imports
import os
os.environ['JAX_ENABLE_X64'] = str(USE_X64).lower()

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [5]:
# Third party
import jax
jax.config.update("jax_disable_jit", not USE_JIT)
jax.config.update("jax_debug_nans", DEBUG_NANS)
import jax.random as jr
import jax.numpy as jnp
from kernax import WhiteNoiseKernel, VarianceKernel, SEKernel, AffineMean

In [6]:
# Local imports
from mimosa.linalg import cho_factor, cho_solve
from mimosa.generate_data import generate_data
from mimosa.hyperpost import hyperpost
from mimosa.sampling import sample_gp
from mimosa.nll import means_nlls, tasks_nlls, full_nll

2026-04-24 14:17:58,573 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/mimosa/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [7]:
# Config
key = jr.PRNGKey(45)

T=120 ; K=3 ; F=1 ; N=75 ; I=1 ; O=2 ; gs=250 if I == 1 else 40

sth=True ; sch=True ; chit=False ; fh=False ; soh=True ; siit=False ; siif=True

mean = AffineMean(slope=0., intercept=0.)
mean_kernel = VarianceKernel(20.) * SEKernel(length_scale=10.)
task_kernel = VarianceKernel(.2) * SEKernel(length_scale=9.) + WhiteNoiseKernel(noise=.01)

mean_priors = {
	"slope": (-.2, .2),
	"intercept": (-2.5, 2.5)
}

mean_kernel_priors = {
	"variance": (5, 10.),
	"length_scale": (2.5, 10.)
}

task_kernel_priors = {
	"variance": (0.25, 1.),
	"length_scale": (2., 8.),
	"noise": (0.01, 0.1)
}

jax.devices()

[CpuDevice(id=0)]

In [8]:
inputs, outputs, maps, grid, m_p_means, m_p_covs, m_p, mix, t_m, m, m_k, t_k = generate_data(
	key, T, K, F, N,  I, O, gs,
	mean, mean_kernel, task_kernel,
	mean_priors, mean_kernel_priors, task_kernel_priors,
	sth, sch, chit, fh, soh, siit, siif)

In [9]:
mix_coeffs = jnp.eye(K)[mix]
mix_coeffs.shape

(120, 3)

In [10]:
p_m, p_c = hyperpost(inputs, outputs, maps, grid, mix_coeffs, m, m_k, t_k)
p_m.shape, p_c.shape

/opt/miniconda3/envs/mimosa/lib/python3.13/site-packages/jax/_src/scipy/linalg.py:167: FutureWarning: jax.scipy.linalg.cho_solve: batched 1D solves with b.ndim > 1 are deprecated, and in the future will be treated as a batched 2D solve. Use cho_solve(c_and_lower, b[..., None]).squeeze(-1) to avoid this warning.
  warnings.warn(


((3, 2, 250), (3, 1, 250, 250))

In [11]:
print(t_k)

VarianceKernel(variance=0.86) * SEKernel(length_scale=7.41) + WhiteNoiseKernel(noise=0.10)


---

## HP Optimisation

Based on:
- An optimiser (LBFG-S, Adam, ...)
- Training config (shared HPs across clusters, tasks, features, outputs, ...)
- Mean processes (post_means, post_covs)
- Mixtures proportions and coefficients
- Trainable parameters (cluster_mean, cluster_kernel, task_kernel, ...)
- Freezing maps for every trainable parameters

We want to provide updated parameters, optimising the cluster and task likelihoods, as fast as possible

In [12]:
import optimistix as optx
import equinox as eqx

In [13]:
train_clust_mean = m.replace(slope=mean.slope, intercept=mean.intercept)
train_clust_kern = m_k.replace(length_scale=mean_kernel.right.length_scale, variance=mean_kernel.left.variance)
train_task_kern = t_k.replace(length_scale=task_kernel.left.right.length_scale, variance=task_kernel.left.left.variance, noise=task_kernel.right.noise)
print(f"{train_clust_mean}\n{train_clust_kern}\n{train_task_kern}")

AffineMean(slope=0.00, intercept=0.00)
VarianceKernel(variance=20.00) * SEKernel(length_scale=10.00)
VarianceKernel(variance=0.20) * SEKernel(length_scale=9.00) + WhiteNoiseKernel(noise=0.01)


In [14]:
true_clust_mean = m
true_clust_kern = m_k
true_task_kern = t_k

print(f"{true_clust_mean}\n{true_clust_kern}\n{true_task_kern}")

AffineMean(slope=-0.07, intercept=-0.53)
VarianceKernel(variance=8.46) * SEKernel(length_scale=5.69)
VarianceKernel(variance=0.86) * SEKernel(length_scale=7.41) + WhiteNoiseKernel(noise=0.10)


### Optimisation

In [15]:
from jax.nn import softmax

def update_mixture(inputs, outputs, mappings, task_kern, post_mean, post_cov, mixture_proportions, jitter=jnp.array(1e-3)):
	task_llhs = tasks_nlls(inputs, outputs, mappings, task_kern(inputs), post_mean, post_cov, jitter=jitter)
	return softmax(jnp.log(mixture_proportions[None, :, None]) - task_llhs, axis=1)

new_mix_coeffs = update_mixture(inputs, outputs, maps, train_task_kern, p_m, p_c, jnp.ones((K,)) / K)

In [16]:
import jax.numpy as jnp
from mimosa.nll import means_nlls, tasks_nlls
import optimistix as optx
from equinox import filter_jit, combine


@filter_jit
def optimise_clusters(
		clust_mean, clust_kern,
		post_mean, post_cov, grid,
		solver=optx.LBFGS(atol=1e-4, rtol=1e-4), jitter=jnp.array(1e-3),
		clust_mean_frozen=None, clust_kern_frozen=None):

	@filter_jit
	def loss_fn(params, frozen):
		mean = params[0] if frozen[0] is None else combine(params[0], frozen[0])
		kern = params[1] if frozen[1] is None else combine(params[1], frozen[1])

		return means_nlls(post_mean, post_cov, grid, mean(grid), kern(grid), jitter=jitter).sum()


	params = (clust_mean, clust_kern)
	frozen = (clust_mean_frozen, clust_kern_frozen)

	return optx.minimise(loss_fn, solver, params, frozen, throw=False)

@filter_jit
def optimise_tasks(
		task_kern,
		inputs, outputs, mappings, post_mean, post_cov, mixture_coeffs,
		solver=optx.LBFGS(atol=1e-4, rtol=1e-4), jitter=jnp.array(1e-3),
		task_kern_frozen=None):

	@filter_jit
	def loss_fn(params, frozen):
		kern = params if frozen is None else combine(params, frozen)

		return (tasks_nlls(inputs, outputs, mappings, kern(inputs), post_mean, post_cov, jitter=jitter) * mixture_coeffs).sum()

	return optx.minimise(loss_fn, solver, task_kern, task_kern_frozen, throw=False)


In [17]:
opti_clust_mean, opti_clust_kern = optimise_clusters(train_clust_mean, train_clust_kern, p_m, p_c, grid).value
print(f"{opti_clust_mean}\n{opti_clust_kern}")

AffineMean(slope=-0.06, intercept=-1.04)
VarianceKernel(variance=7.80) * SEKernel(length_scale=5.68)


### Task optimisation

In [18]:
opti_task_kern = optimise_tasks(train_task_kern, inputs, outputs, maps, p_m, p_c, new_mix_coeffs).value
print(f"{opti_task_kern}")

VarianceKernel(variance=0.83) * SEKernel(length_scale=7.32) + WhiteNoiseKernel(noise=0.10)


In [23]:
task_kernel_params = eqx.tree_at(lambda k: k.inner.inner.inner.right._noise, train_task_kern, None)
task_kernel_frozen = eqx.tree_at(lambda k: k.inner.inner.inner.right._noise, train_task_kern, train_task_kern.inner.inner.inner.right._noise_parametrisation.wrap(0.1))

In [24]:
print(f"{task_kernel_params}\n{task_kernel_frozen}")

VarianceKernel(variance=0.20) * SEKernel(length_scale=9.00) + WhiteNoiseKernel()
VarianceKernel(variance=0.20) * SEKernel(length_scale=9.00) + WhiteNoiseKernel(noise=0.10)


In [25]:
opti_task_kern_1 = optimise_tasks(task_kernel_params, inputs, outputs, maps, p_m, p_c, new_mix_coeffs, task_kern_frozen=task_kernel_frozen).value
print(f"{eqx.combine(opti_task_kern_1, task_kernel_frozen)}")

VarianceKernel(variance=0.84) * SEKernel(length_scale=7.35) + WhiteNoiseKernel(noise=0.10)
